# ==============================================================================
# 📱 CONSUMER ELECTRONICS: MOBILE PRICE RANGE MULTICLASS VIA RANDOM FOREST
# ==============================================================================
### Core Principles:
Classifying smartphones into 4 distinct price tiers (0=Low, 1=Medium, 2=High, 3=Very High) using 20 hardware features (`battery_power`, `ram`, `px_height`, `px_width`).
Random Forest achieves high accuracy by aggregating votes across multi-tier hardware partitions.


In [1]:
# STEP 1: ENVIRONMENT SETUP
import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("✅ STEP 1: Mobile Electronics Analytics Environment ready.")


✅ STEP 1: Mobile Electronics Analytics Environment ready.


## 📥 STEP 2 & 3: INGESTION & DATASET EXPLORATION
Loading Kaggle Mobile Price Classification dataset (2,000 devices).


In [2]:
# STEP 2 & 3: LOAD & SANITIZE
data_path = 'data/mobile_price/train.csv'
df = pd.read_csv(data_path)

df.columns = [c.strip().lower() for c in df.columns]
print(f"📊 Dataset Shape: {df.shape[0]:,} mobile devices, {df.shape[1]} attributes")
print(f"Price range distribution: {df['price_range'].value_counts().to_dict()}")
df.head(3)


📊 Dataset Shape: 2,000 mobile devices, 21 attributes
Price range distribution: {1: 500, 2: 500, 3: 500, 0: 500}


,battery_power,blue,clock_speed,dual_sim,fc,four_g,int_memory,m_dep,mobile_wt,n_cores,...,px_height,px_width,ram,sc_h,sc_w,talk_time,three_g,touch_screen,wifi,price_range
0,842,0,2.2,0,1,0,7,0.6,188,2,...,20,756,2549,9,7,19,0,0,1,1
1,1021,1,0.5,1,0,1,53,0.7,136,3,...,905,1988,2631,17,3,7,1,1,0,2
2,563,1,0.5,1,2,1,41,0.9,145,5,...,1263,1716,2603,11,2,9,1,1,0,2


## 🧹 STEP 4 & 5: SEGREGATION & HYGIENE
Separating features and target `price_range`.


In [3]:
# STEP 4 & 5: SEGREGATION & HYGIENE
X = df.drop(columns=['price_range'])
y = df['price_range']

print(f"Missing attributes: {X.isnull().sum().sum()}")
print(f"Class Balance (4 Tiers):\n{y.value_counts(normalize=True).round(4) * 100}")


Missing attributes: 0
Class Balance (4 Tiers):
price_range
1    25.0
2    25.0
3    25.0
0    25.0
Name: proportion, dtype: float64


## 🔒 STEP 6: STRATIFIED SPLIT
80/20 train/test split.


In [4]:
# STEP 6: STRATIFIED SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Train Devices: {X_train.shape[0]:,}")
print(f"Test Devices : {X_test.shape[0]:,}")


Train Devices: 1,600
Test Devices : 400


## ⚙️ STEP 7: SCALE-INVARIANT PREPROCESSOR
All 20 features are numerical hardware specs; scale-invariant pass-through.


In [5]:
# ==============================================================================
# ⚙️ STEP 7: TREE-OPTIMIZED UNIVERSAL PREPROCESSOR (SCALE-INVARIANT)
# ==============================================================================
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

def build_tree_preprocessor(X):
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = X.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
    transformers = []
    if len(num_cols) > 0:
        transformers.append(('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), num_cols))
    if len(cat_cols) > 0:
        transformers.append(('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('ohe', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False))
        ]), cat_cols))
    return ColumnTransformer(transformers=transformers, remainder='drop') if len(transformers) > 0 else 'passthrough'

preprocessor = build_tree_preprocessor(X_train)
print(f"✅ STEP 7: Tree-Optimized Universal Preprocessor Ready.")

✅ STEP 7: Tree-Optimized Universal Preprocessor Ready.


## ⚠️ STEP 8: BASELINE ENSEMBLE EVALUATION
Measuring Out-of-Bag validation accuracy.


In [6]:
# STEP 8: UNCONSTRAINED BASELINE FIT (OVERFITTING PROOF)
pipe_rf = Pipeline([
    ('prep', preprocessor),
    ('rf', RandomForestClassifier(n_estimators=150, oob_score=True, random_state=42, n_jobs=-1))
])
pipe_rf.fit(X_train, y_train)

acc_test = accuracy_score(y_test, pipe_rf.predict(X_test))
oob_acc = pipe_rf.named_steps['rf'].oob_score_

print(f"Random Forest OOB Accuracy : {oob_acc*100:.2f}%")
print(f"Random Forest Test Accuracy: {acc_test*100:.2f}%")

Random Forest OOB Accuracy : 86.88%
Random Forest Test Accuracy: 87.25%


## 🎯 STEP 9: HYPERPARAMETER TUNING VIA 5-FOLD CV
Optimizing `max_depth` and `max_features` for multiclass accuracy.


In [7]:
# STEP 9: GRID SEARCH
param_grid = {
    'rf__n_estimators': [100, 150],
    'rf__max_depth': [10, 14, 18],
    'rf__max_features': ['sqrt', 0.5, 0.7]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
pipe_tune = Pipeline([
    ('prep', preprocessor),
    ('rf', RandomForestClassifier(random_state=42, n_jobs=-1))
])
grid = GridSearchCV(pipe_tune, param_grid=param_grid, cv=cv, scoring='accuracy', n_jobs=-1)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
print(f"🏆 Best Hyperparameters: {grid.best_params_}")
print(f"🎯 Best 5-Fold CV Accuracy: {grid.best_score_*100:.2f}%")

🏆 Best Hyperparameters: {'rf__max_depth': 18, 'rf__max_features': 0.5, 'rf__n_estimators': 100}
🎯 Best 5-Fold CV Accuracy: 89.13%


## ⚔️ STEP 10: ALGORITHM DUEL (Random Forest vs Single Tree)
Evaluating ensemble multiclass gain.


In [8]:
# STEP 10: BENCHMARK
from sklearn.tree import DecisionTreeClassifier

pipe_single = Pipeline([
    ('prep', preprocessor),
    ('tree', DecisionTreeClassifier(random_state=42))
])
pipe_single.fit(X_train, y_train)

acc_tree = accuracy_score(y_test, pipe_single.predict(X_test))
acc_rf = accuracy_score(y_test, best_model.predict(X_test))

print("="*65)
print(f"Single Decision Tree Accuracy : {acc_tree*100:.2f}%")
print(f"Random Forest (Tuned) Accuracy: {acc_rf*100:.2f}% (+{(acc_rf - acc_tree)*100:.2f}% Gain!)")
print("="*65)

Single Decision Tree Accuracy : 83.00%
Random Forest (Tuned) Accuracy: 92.75% (+9.75% Gain!)


## 📊 STEP 11: MULTICLASS EVALUATION REPORT & CONFUSION MATRIX
Reviewing 4-tier price prediction precision and recall.


In [9]:
# STEP 11: FEATURE IMPORTANCE & ERROR DIAGNOSTICS
y_pred_rf = best_model.predict(X_test)
print("📋 MOBILE PRICE TIER CLASSIFICATION REPORT:")
print(classification_report(y_test, y_pred_rf, target_names=['Budget (0)', 'Economy (1)', 'Premium (2)', 'Flagship (3)']))

cm = confusion_matrix(y_test, y_pred_rf)
print(f"Confusion Matrix:\n{cm}")


📋 MOBILE PRICE TIER CLASSIFICATION REPORT:
              precision    recall  f1-score   support

  Budget (0)       0.95      0.98      0.97       100
 Economy (1)       0.92      0.88      0.90       100
 Premium (2)       0.88      0.90      0.89       100
Flagship (3)       0.96      0.95      0.95       100

    accuracy                           0.93       400
   macro avg       0.93      0.93      0.93       400
weighted avg       0.93      0.93      0.93       400

Confusion Matrix:
[[98  2  0  0]
 [ 5 88  7  0]
 [ 0  6 90  4]
 [ 0  0  5 95]]


## 💾 STEP 12 & 13: PRODUCTION SERIALIZATION & E-COMMERCE SMOKE TEST
Deploying model pipeline and validating instant catalog pricing latency.


In [10]:
# STEP 12 & 13: PRODUCTION SERIALIZATION & LATENCY SMOKE TEST
os.makedirs('production_models', exist_ok=True)
model_path = 'production_models/mobile_price_random_forest_pipeline.joblib'
joblib.dump(best_model, model_path, compress=3)
print(f"💾 Production pipeline saved to: {model_path} ({os.path.getsize(model_path):,} bytes)")

loaded_pipe = joblib.load(model_path)
sample_device = X_test.iloc[[0]]

t0 = time.perf_counter()
pred_tier = loaded_pipe.predict(sample_device)[0]
tier_probs = loaded_pipe.predict_proba(sample_device)[0]
latency_ms = (time.perf_counter() - t0) * 1000

tiers = ['Budget ($)', 'Economy ($$)', 'Premium ($$$)', 'Flagship ($$$$)']
print(f"\n📱 LIVE CATALOG TIER SPECIFICATION AUDIT:")
print(f"   Predicted Price Category: 🏷️ {tiers[pred_tier]}")
print(f"   Model Confidence        : {tier_probs.max()*100:.2f}%")
print(f"   Catalog Triage Latency  : {latency_ms:.3f} ms (SLA < 2.0ms: {'PASS' if latency_ms < 2.0 else 'CHECK'})")


💾 Production pipeline saved to: production_models/mobile_price_random_forest_pipeline.joblib (536,172 bytes)



📱 LIVE CATALOG TIER SPECIFICATION AUDIT:
   Predicted Price Category: 🏷️ Flagship ($$$$)
   Model Confidence        : 95.00%
   Catalog Triage Latency  : 75.391 ms (SLA < 2.0ms: CHECK)
